# 🧠 Taller de Cortex AI Functions en Snowflake
### 📝 Versión para el Alumno


---

### El share de este taller

Usamos **`SFC_SAMPLES.SAMPLE_DATA`**, el share público de Snowflake con el dataset **TPC-H**.


## 📦 Tablas en `TPCH_SF1`

| Tabla | Filas aprox. | Descripción |
|---|---|---|
| `ORDERS` | 1.5 M | Pedidos |
| `CUSTOMER` | 150 K | Clientes |
| `PART` | 200 K | Catálogo de productos |
| `SUPPLIER` | 10 K | Proveedores |
| `NATION` / `REGION` | 25 / 5 | Geografía |

---
## ⚙️ Configuración

Ejecuta esta celda primero. El rol activo necesita el database role `SNOWFLAKE.CORTEX_USER`.

In [ ]:
USE ROLE CURSO_DATA_ENGINEERING;
USE WAREHOUSE WH_CURSO_DATA_ENGINEERING;
USE DATABASE SNOWFLAKE_SAMPLE_DATA;
USE SCHEMA TPCH_SF1;

SHOW TABLES IN SCHEMA SNOWFLAKE_SAMPLE_DATA.TPCH_SF1;

---
## 🔍 Ejercicio 0 — Exploración de datos

Antes de usar funciones AI, familiarízate con los comentarios disponibles en `ORDERS`.

In [ ]:
SELECT O_ORDERKEY, O_ORDERSTATUS, O_TOTALPRICE, O_ORDERDATE, O_ORDERPRIORITY, O_COMMENT
FROM   ORDERS
WHERE  O_COMMENT IS NOT NULL AND LENGTH(O_COMMENT) > 30
LIMIT  10;

---
## 🏋️ Ejercicio 1 — `AI_SENTIMENT`

### Análisis de sentimiento por aspectos

`AI_SENTIMENT` va más allá del score numérico: devuelve **sentimiento por categorías** que tú defines (entrega, precio, calidad…), además del global.

Retorna un objeto JSON: `{ "categories": [ {"name":"overall","sentiment":"mixed"}, ... ] }`

**Tarea:**
1. Analiza 20 comentarios recientes de `ORDERS`.
2. Extrae sentimiento global y desglose por `delivery`, `price` y `quality`.
3. Filtra solo los comentarios con sentimiento global `negative` o `mixed`.

> 💡 **Sintaxis:** `AI_SENTIMENT(<texto>, ['aspecto1', 'aspecto2'])` — los aspectos son opcionales
>
> 💡 **Extraer del JSON:** `resultado:categories[0]:sentiment::STRING`
>
> El índice 0 siempre es `overall`. Los aspectos siguen en orden: índice 1, 2, 3…
>
> Valores posibles: `positive`, `negative`, `neutral`, `mixed`, `unknown`

In [ ]:
-- Ejercicio 1: AI_SENTIMENT con aspectos
-- Columnas esperadas: O_ORDERKEY, O_ORDERPRIORITY, O_COMMENT,
--                     SENT_GLOBAL, ASPECTO_DELIVERY, ASPECTO_PRICE, ASPECTO_QUALITY
-- Tabla: ORDERS | Filtro: comentarios no nulos > 20 chars | Orden por fecha DESC | Límite: 20
-- Filtra en el SELECT exterior: solo registros con SENT_GLOBAL IN ('negative','mixed')
-- Pista: usa una CTE para guardar el JSON y parsea en el SELECT exterior



---
## 🏋️ Ejercicio 2 — `AI_TRANSLATE`

### Traducción con detección automática de idioma

`AI_TRANSLATE`  el **idioma origen es opcional** — puedes omitirlo y Snowflake lo detecta automáticamente.

**Tarea:**
1. Toma 10 comentarios de `PART` (campo `P_COMMENT`).
2. Tradúcelos al español sin especificar idioma origen.
3. Muestra nombre del producto, comentario original y traducción.

> 💡 **Sintaxis:** `AI_TRANSLATE(<texto>, '<idioma_destino>')`
>
> Con idioma origen explícito: `AI_TRANSLATE(<texto>, '<destino>', '<origen>')`
>
> Códigos: `es` español · `en` inglés · `fr` francés · `de` alemán · `pt` portugués · `ja` japonés

In [ ]:
-- Ejercicio 2: AI_TRANSLATE
-- Columnas esperadas: P_PARTKEY, P_NAME, P_MFGR, COMENTARIO_ORIGINAL, COMENTARIO_EN_ESPANOL
-- Tabla: PART | Filtro: comentarios no nulos con longitud entre 20 y 120 chars | Límite: 10



---
## 🏋️ Ejercicio 3 — `AI_SUMMARIZE_AGG`

### Resumen agregado sin límite de contexto

`AI_SUMMARIZE_AGG` es una **función de agregación nativa** — funciona igual que `COUNT()` o `SUM()`, pero sobre texto. No tiene límite de ventana de contexto.

**Tarea:**
1. Identifica los 5 clientes con mayor gasto total.
2. Para cada uno, usa `AI_SUMMARIZE_AGG` en un `GROUP BY` para resumir todos sus comentarios de pedido.
3. Muestra cliente, gasto y el resumen generado.

> 💡 **Sintaxis:** `AI_SUMMARIZE_AGG(<columna_texto>)` — úsalo como cualquier función de agregación
>
> 💡 Pista: usa una CTE para identificar los top 5 y luego el GROUP BY con la función

In [ ]:
-- Ejercicio 3: AI_SUMMARIZE_AGG como función de agregación
-- Columnas esperadas: C_NAME, GASTO_TOTAL_USD, RESUMEN_ACTIVIDAD
-- Tablas: ORDERS + CUSTOMER | Top 5 clientes por SUM(O_TOTALPRICE)
-- Pista: CTE para top 5 → JOIN con CUSTOMER y ORDERS → GROUP BY cliente con AI_SUMMARIZE_AGG



---
## 🏋️ Ejercicio 4 — `AI_CLASSIFY`

### Clasificación con ejemplos few-shot y multi-label

`AI_CLASSIFY`
- **`examples`** por categoría → mejora precisión con ejemplos concretos (*few-shot*)
- **`output_mode: 'multi'`** → un texto puede recibir varias categorías a la vez

**Tarea:**
1. Toma 15 comentarios de `SUPPLIER`.
2. Clasifícalos en estas categorías de negocio: `logistica`, `calidad`, `precio`, `servicio`, `otro`.
3. Incluye `description` y al menos un `example` por categoría.
4. Extrae label y score de confianza del JSON resultado.

> 💡 **Sintaxis:**
> ```sql
> AI_CLASSIFY(
>   <texto>,
>   [ {'label':'cat', 'description':'...', 'examples':['ej1','ej2']}, ... ],
>   {'output_mode': 'single'}
> )
> ```
> Devuelve JSON: `{"label":"...", "score":0.95}` en modo single
>
> 💡 Pista: usa una CTE para guardar el JSON y extrae `resultado:label::STRING` en el SELECT exterior

In [ ]:
-- Ejercicio 4: AI_CLASSIFY con descripciones y ejemplos few-shot
-- Columnas esperadas: S_SUPPKEY, S_NAME, S_COMMENT, CATEGORIA, CONFIANZA
-- Tabla: SUPPLIER | Filtro: comentarios no nulos > 15 chars | Límite: 15 | Orden por CONFIANZA DESC
-- Pista: CTE para clasificar → SELECT exterior para parsear el JSON



---
## 🏋️ Ejercicio 5 — `AI_COMPLETE` + `PROMPT()`

### Generación de texto con modelos de última generación

`AI_COMPLETE` 
- Modelos actualizados: `claude-sonnet-4-5`, `gpt-5`, `llama4-maverick`, `gemini-2-5-flash`…
- El helper **`PROMPT('texto con {0} placeholders', col1, col2)`** construye prompts dinámicos sin concatenar strings.
- Parámetros opcionales: `{'temperature': 0.3, 'max_tokens': 200}`

**Tarea:**
1. Selecciona los 5 pedidos urgentes de mayor valor (join `ORDERS` + `CUSTOMER`).
2. Usa `AI_COMPLETE` + `PROMPT()` incluyendo cliente, fecha, importe, estado y comentario.
3. Genera una nota ejecutiva de seguimiento en español, máx. 50 palabras por pedido.

> 💡 **Sintaxis:** `AI_COMPLETE('<modelo>', PROMPT('texto {0}', columna))`
>
> Con parámetros: `AI_COMPLETE('<modelo>', prompt_obj, {'temperature': 0.3, 'max_tokens': 120})`
>
> 💡 Pista: CTE para preparar datos → AI_COMPLETE en el SELECT exterior

In [ ]:
-- Ejercicio 5: AI_COMPLETE con PROMPT()
-- Columnas esperadas: O_ORDERKEY, CLIENTE, FECHA, IMPORTE_USD, ESTADO, NOTA_EJECUTIVA
-- Tablas: ORDERS + CUSTOMER | Filtro: O_ORDERPRIORITY = '1-URGENT' | Orden por importe DESC | Límite: 5
-- Usa COALESCE(O_COMMENT, 'sin comentario') para el comentario
-- Pista: CTE para los 5 pedidos → AI_COMPLETE con PROMPT() en el SELECT final



---
## 🏋️ Ejercicio 6 — `AI_FILTER`

### Filtrado semántico en cláusula WHERE

`AI_FILTER`  devuelve `TRUE`/`FALSE` evaluando una condición en lenguaje natural sobre texto. Úsala en `SELECT`, `WHERE` o `JOIN ... ON`.

Reemplaza búsquedas con `ILIKE '%palabra%'` que no capturan sinónimos ni contexto implícito.

**Tarea:**
1. Filtra proveedores (`SUPPLIER` + join con `NATION`) cuyo comentario mencione **problemas de retraso o entrega pendiente**.
2. En el SELECT muestra también el resultado de `ILIKE '%delay%' OR ILIKE '%late%'` para comparar.
3. Aplica `AI_FILTER` directamente en el `WHERE`.

> 💡 **Sintaxis:** `AI_FILTER('<condición en lenguaje natural>')`
>
> Usable en: `WHERE AI_FILTER(...)` · `SELECT AI_FILTER(...) AS flag` · `JOIN ... ON AI_FILTER(...)`
>
> 💡 No necesita CTE — puedes llamarla dos veces (en SELECT y WHERE) con la misma condición

In [ ]:
-- Ejercicio 6: AI_FILTER — filtrado semántico
-- Columnas esperadas: S_SUPPKEY, S_NAME, PAIS, S_COMMENT, ILIKE_MATCH
-- Tablas: SUPPLIER + NATION | Filtro: comentarios no nulos > 15 chars
-- Aplica AI_FILTER en el WHERE para quedarte solo con problemas de entrega/retraso
-- Añade columna ILIKE_MATCH para comparar con búsqueda clásica | Límite: 20



---
## 🏋️ Ejercicio 7 — `AI_EXTRACT`

### Extracción multi-campo en una sola llamada

`AI_EXTRACT`  La mejora clave: soporta **múltiples preguntas en una llamada** devolviendo un JSON con todas las respuestas.

**Tarea:**
1. Usa comentarios de `SUPPLIER` (join con `NATION`).
2. En **una sola llamada** a `AI_EXTRACT`, extrae tres campos: el problema, el producto afectado y si hay urgencia.
3. Filtra las filas donde el campo `problema` no esté vacío.

> 💡 **Sintaxis multi-campo:**
> ```sql
> AI_EXTRACT(
>   <texto>,
>   {'campo1': '<pregunta1>', 'campo2': '<pregunta2>', ...}
> )
> ```
> Devuelve un OBJECT JSON — extrae con `resultado:response:campo1::STRING`
>
> 💡 Pista: CTE para guardar el JSON → SELECT exterior para parsear y filtrar

In [ ]:
-- Ejercicio 7: AI_EXTRACT con múltiples campos
-- Columnas esperadas: S_SUPPKEY, S_NAME, PAIS, CONTEXTO, PROBLEMA_DETECTADO, PRODUCTO_AFECTADO, HAY_URGENCIA
-- Tablas: SUPPLIER + NATION | Filtro: comentarios no nulos > 20 chars | Límite en CTE: 30
-- Filtra en SELECT exterior donde PROBLEMA_DETECTADO no sea NULL ni vacío | Orden: S_NAME



---
## 🏋️ Ejercicio 8 — `AI_AGG`

### Agregación con instrucción personalizada

`AI_AGG`  Se diferencia de `AI_SUMMARIZE_AGG` en que tú defines exactamente qué insight quieres extraer del grupo — como darle una instrucción específica a un analista.

**Tarea:**
Para cada prioridad de pedido (`O_ORDERPRIORITY`), usa `AI_AGG` con un prompt que devuelva:
- Los 2-3 temas recurrentes
- El tono predominante (positivo/negativo/neutro)
- Una recomendación de acción en una frase

> 💡 **Sintaxis:** `AI_AGG(<columna_texto>, '<instrucción_de_análisis>')`
>
> La instrucción puede pedir formato estructurado. Sin límite de filas por grupo.
>
> 💡 Pista: usa `QUALIFY ROW_NUMBER() OVER (PARTITION BY O_ORDERPRIORITY ORDER BY RANDOM()) <= 50` para muestrear y controlar el coste

In [ ]:
-- Ejercicio 8: AI_AGG con instrucción personalizada
-- Columnas esperadas: O_ORDERPRIORITY, COMENTARIOS_ANALIZADOS, ANALISIS_EJECUTIVO
-- Tabla: ORDERS | Filtro: comentarios no nulos, fecha >= '1997-01-01'
-- Muestrea 50 filas por grupo con QUALIFY ROW_NUMBER() <= 50 para controlar costes
-- La instrucción al AI_AGG debe pedir: temas recurrentes, tono predominante y recomendación en español
-- GROUP BY O_ORDERPRIORITY | Orden: O_ORDERPRIORITY



---
## 📊 Resumen de funciones `AI_*` del taller

| Función | Tipo | Resumen clave |
|---|---|---|
| `AI_SENTIMENT` | Task-specific | Análisis por aspectos, multilingüe |
| `AI_TRANSLATE` | Task-specific  | Detección automática de idioma origen |
| `AI_SUMMARIZE_AGG` | Agregación  | Función agregada nativa, sin límite contexto |
| `AI_CLASSIFY` | Task-specific  | Few-shot examples, multi-label, imágenes |
| `AI_COMPLETE` | Generativa | Modelos actualizados, helper `PROMPT()` |
| `AI_FILTER` | Task-specific | Filtrado semántico booleano en WHERE/JOIN |
| `AI_EXTRACT` | Task-specific  | Multi-campo en una llamada |
| `AI_AGG` | Agregación  | Insight personalizado sin límite de contexto |

### 🔗 Documentación oficial
- [Cortex AI Functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)
- [AI_SENTIMENT con aspectos](https://docs.snowflake.com/en/sql-reference/functions/ai_sentiment)
- [AI_COMPLETE + modelos disponibles](https://docs.snowflake.com/en/sql-reference/functions/ai_complete)
- [AI_FILTER](https://docs.snowflake.com/en/sql-reference/functions/ai_filter)
- [Data Sharing](https://docs.snowflake.com/en/user-guide/data-sharing-intro)

> ✅ **Controlar costes:** Usa `AI_COUNT_TOKENS('ai_complete', texto, 'modelo')` para estimar tokens antes de queries masivas. Añade `QUALIFY ROW_NUMBER() ... <= N` para muestrear por grupo.